# Core MCP Feature

## 1. Sampling

Sampling allows **a server to access a language model like Claude through a connected MCP client**. Instead of the server directly calling Claude, it asks the client to make the call on its behalf. This shifts the responsibility and cost of text generation from the server to the client.

![](./figure/09/Sampling_03.png)
![](./figure/09/Sampling_04.png)

### The Problem Sampling Solves

Imagine you have an MCP server with a research tool that fetches information from Wikipedia. After gathering all that data, you need to summarize it into a coherent report. You have two options:

1. Give the MCP server direct access to Claude. The server would need *its own API key, handle authentication, manage costs, and implement all the Claude integration code*. This works but **adds significant complexity**.

2. Use sampling. The server generates a prompt and asks the client "Could you call Claude for me?" The client, which already has a connection to Claude, makes the call and returns the results.

![](./figure/09/Sampling_01.png)
![](./figure/09/Sampling_02.png)

### How Sampling Works

The flow is straightforward:

1. Server completes its work (like fetching Wikipedia articles)
2. Server creates a prompt asking for text generation
3. Server sends a sampling request to the client
4. Client calls Claude with the provided prompt
5. Client returns the generated text to the server
6. Server uses the generated text in its response

### Benefits of Sampling

- **Reduces server complexity** : The server doesn't need to integrate with language models directly
- **Shifts cost burden** : The client pays for token usage, not the server
- **No API keys needed** : The server doesn't need credentials for Claude
- **Perfect for public servers** : You don't want a public server racking up AI costs for every user

### Implementation

Setting up sampling requires code on both sides:

#### 1. Server Side
In your tool function, use the `create_message` function to request text generation:

```python
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
    Please summarize the following text:
    {text_to_summarize}
    """
    
    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user",
                content=TextContent(
                    type="text",
                    text=prompt
                )
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant",
    )
    
    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")
```

#### 2. Client Side
Create a sampling callback that handles the server's requests:

```python
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)
    
    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )
```

Then pass this callback when initializing your client session:

```python
async with ClientSession(
    read,
    write,
    sampling_callback=sampling_callback
) as session:
    await session.initialize()
```

### When to Use Sampling

Sampling is most valuable when building publicly accessible MCP servers. You don't want random users generating unlimited text at your expense. By using sampling, each client pays for their own AI usage while still benefiting from your server's functionality.

The technique essentially moves the AI integration complexity from your server to the client, which often already has the necessary connections and credentials in place.

## 為什麼需要 Sampling？

你的理解是正確的——**大多數 tool use 情境**確實不需要 sampling。但有一類特殊情境會讓這個架構不夠用。

---

### 一般 Tool Use 架構

```
Client → [呼叫 LLM] → LLM 決定用 tool → Client 執行 tool → Client 把結果送回 LLM → 完成
```

這個流程的關鍵：**Client 全程掌控**，LLM 只是被動回應。

---

### 問題出現在哪？—— **MCP Server 需要「自己思考」的時候**

想像這個情境：

> 你寫了一個 **MCP Server**，功能是「自動幫你 code review 整個 repo，發現問題就開 GitHub Issue」

這個 server 需要：
1. 讀取每一個檔案
2. **理解程式碼、判斷有沒有問題**（這需要 LLM 的推理能力）
3. 如果有問題，自動開 Issue

**問題來了：MCP Server 本身沒有呼叫 LLM 的能力！**

```
Client (有 LLM 連線)
    ↓ 呼叫 tool
MCP Server (只是個程式，沒有 LLM)
    ↓ 讀了檔案，但看不懂程式碼
    ↓ ??? 我要怎麼判斷這段 code 有沒有問題
```

---

### Sampling 解決了什麼？

Sampling 讓 **MCP Server 可以反過來請 Client 幫它呼叫 LLM**：

```
Client (有 LLM 連線)
    ↓ 呼叫 tool
MCP Server
    ↓ 讀取 file_a.py
    ↓ 「我需要 LLM 幫我分析這段 code」
    ↑ 發出 sampling request（帶著 code 內容）
Client
    ↓ 收到 sampling request，呼叫 LLM
    ↓ 把 LLM 的分析結果還給 MCP Server
MCP Server
    ↓ 拿到分析結果，決定要不要開 Issue
    ↓ 繼續處理下一個檔案...
```

---

### 具體對比

| | 一般 Tool Use | 需要 Sampling |
|---|---|---|
| **誰在思考** | LLM（Client 端） | MCP Server 需要自己做複雜判斷 |
| **流程** | LLM → 呼叫 tool → 拿結果 | Server 執行中途需要 LLM 協助 |
| **例子** | 查天氣、查資料庫 | 分析文件、多步驟 agent 任務 |

---

### 一句話總結

> 一般 tool use 是「**LLM 叫 Server 做事**」；  
> Sampling 是當 **Server 在做事的過程中，需要反過來請 LLM 幫忙思考**。

本質上是為了讓 MCP Server 能夠執行**需要 AI 推理的複雜 agentic 任務**，而不只是執行固定邏輯的程式。

## 2. Sampling Walkthrough
code 在 [./sampling](./sampling/)

### Initiating sampling

On the server, during a tool call, run the `create_message()` method, passing in some messages that you wish to send to a language model.

```python
# server.py
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
        Please summarize the following text:
        {text_to_summarize}
    """

    ''' ============================================================================ '''
    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user", content=TextContent(type="text", text=prompt)
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant.",
    )
    ''' ============================================================================ '''

    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")
```

### Sampling callback

On the client, you must implement a sampling callback. It will receive a list of messages provided by the server.

```python
# client.py
''' ============================================================================ '''
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
''' ============================================================================ '''
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)

    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )
```

### Message formats

The list of messages provided by the server are formatted for communication in MCP. The individual messages aren't guaranteed to be compatible with whatever LLM SDK you are using.

For example, if you're using the Anthropic SDK, you'll have to write a little bit of conversion logic to turn the MCP messages into a format compatible with Anthropic's SDK.
> 看使用哪個 LLM，需要將其調整為該 model 能接受的 format

```python
# client.py
async def chat(input_messages: list[SamplingMessage], max_tokens=4000):
    messages = []
    ''' ============================================================================ '''
    for msg in input_messages:
        if msg.role == "user" and msg.content.type == "text":
            content = (
                msg.content.text
                if hasattr(msg.content, "text")
                else str(msg.content)
            )
            messages.append({"role": "user", "content": content})
        elif msg.role == "assistant" and msg.content.type == "text":
            content = (
                msg.content.text
                if hasattr(msg.content, "text")
                else str(msg.content)
            )
            messages.append({"role": "assistant", "content": content})
    ''' ============================================================================ '''
    response = await anthropic_client.messages.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
    )

    text = "".join([p.text for p in response.content if p.type == "text"])
    return text
```

### Returning generated text

After generating text with the LLM, you'll return a `CreateMessageResult`, which contains the generated text.

```python
# client.py
async def sampling_callback(
    context: RequestContext, params: CreateMessageRequestParams
):
    ''' ============================================================================ '''
    # Call Claude using the Anthropic SDK
    text = await chat(params.messages)

    return CreateMessageResult(
        role="assistant",
        model=model,
        content=TextContent(type="text", text=text),
    )
    ''' ============================================================================ '''
```

### Connecting the callback

Don't forget: the callback on the client needs to be passed into the `ClientSession` call.

```python
# client.py
async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            ''' ============================================================================ '''
            read, write, sampling_callback=sampling_callback
            ''' ============================================================================ '''
        ) as session:
            await session.initialize()

            result = await session.call_tool(
                name="summarize",
                arguments={"text_to_summarize": "lots of text"},
            )
            print(result.content)
```

### Getting the result

After the client has generated and returned some text, it will be sent to the server. You can do anything with this text:

- Use it as part of a workflow in your tool
- Decide to make another sampling call
- Return the generated text

```python
# server.py
@mcp.tool()
async def summarize(text_to_summarize: str, ctx: Context):
    prompt = f"""
        Please summarize the following text:
        {text_to_summarize}
    """

    result = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user", content=TextContent(type="text", text=prompt)
            )
        ],
        max_tokens=4000,
        system_prompt="You are a helpful research assistant.",
    )
    ''' ============================================================================ '''
    if result.content.type == "text":
        return result.content.text
    else:
        raise ValueError("Sampling failed")
    ''' ============================================================================ '''
```

### 路線圖

In [ ]:
[Server]                              [Client]
  ↓
`summarize()` 被呼叫
  ↓
`ctx.session.create_message()`  ──────→  MCP 協議路由 (知道要轉交給 `sampling_callback()` 處理，因為在 `ClientSession(..., sampling_callback=sampling_callback)` 有註冊)
(MCP 框架內部發出                            |
`sampling/createMessage` request)           |
  ↓ (await，暫停等待)                        ↓
                                      `sampling_callback()` 被自動呼叫
                                            ↓
                                      `chat()` 轉換格式 + 呼叫 Anthropic API
                                            ↓
                                      return `CreateMessageResult(...)`
  ↓ (await 結束，拿到 result)  ←──────    MCP 協議路由回傳
`result.content.text`
  ↓
return 給最初呼叫者

## 3. Log and progress notifications

Logging and progress notifications are simple to implement but make a huge difference in **User Experience** when working with MCP servers. They help users understand what's happening during long-running operations instead of wondering if something has broken.

When Claude calls a tool that takes time to complete - like researching a topic or processing data - users typically see nothing until the operation finishes. This can be frustrating because they don't know if the tool is working or has stalled.

With logging and progress notifications enabled, users get real-time feedback showing exactly what's happening behind the scenes. They can see progress bars, status messages, and detailed logs as the operation runs.

![](./figure/09/log_and_progresss_01.png)
![](./figure/09/log_and_progresss_02.png)

### How It Works

In the Python MCP SDK, logging and progress notifications work through the `Context` argument that's **automatically provided to your tool functions**. This context object gives you methods to communicate back to the client during execution.

The key methods you'll use are:

1. `context.info()` : Send log messages to the client
2. `context.report_progress()` : Update progress with current and total values

```python
@mcp.tool(
    name="research",
    description="Research a given topic"
)
async def research(
    topic: str = Field(description="Topic to research"),
    *,
    context: Context
):
    await context.info("About to do research...") # <--
    await context.report_progress(20, 100) # <--
    sources = await do_research(topic)
    
    await context.info("Writing report...") # <--
    await context.report_progress(70, 100) # <--
    results = await generate_report(sources)
    
    return results
```

### Client-Side Implementation

On the client side, you need to set up *callback* functions to handle these notifications. The server emits these messages, but it's up to your client application to decide **how to present** them to users.

You provide the **logging callback** when *creating the client session*, and the **progress callback** when *making individual tool calls*. This gives you flexibility to handle different types of notifications appropriately.

```python
async def logging_callback(params: LoggingMessageNotificationParams):
    print(params.data)

async def print_progress_callback(
    progress: float, total: float | None, message: str | None
):
    if total is not None:
        percentage = (progress / total) * 100
        print(f"Progress: {progress}/{total} ({percentage:.1f}%)")
    else:
        print(f"Progress: {progress}")

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read,
            write,
            logging_callback=logging_callback # <-- Setup log message
        ) as session:
            await session.initialize()
            
            await session.call_tool(
                name="add",
                arguments={"a": 1, "b": 3},
                progress_callback=print_progress_callback, # <-- Setup progress message
            )
```

### Presentation Options

How you present these notifications depends on your application type:

- **CLI applications** : Simply print messages and progress to the terminal
- **Web applications** : Use WebSockets, server-sent events, or polling to push updates to the browser
- **Desktop applications** : Update progress bars and status displays in your UI

Remember that implementing these notifications is entirely optional. You can choose to ignore them completely, show only certain types, or present them however makes sense for your application. They're purely user experience enhancements to help users understand what's happening during long-running operations.

## 4. Notifications walkthrough
code 在 [./notifications](./notifications)

### Tool function receives Context argument

Tool functions automatically receive `Context` as their last argument. This object has methods for logging and reporting progress to the client.

```python
# server.py
@mcp.tool()
'''================================================='''
async def add(a: int, b: int, ctx: Context) -> int:
'''================================================='''
    await ctx.info("Preparing to add...")
    await ctx.report_progress(20, 100)

    await asyncio.sleep(2)

    await ctx.info("OK, adding...")
    await ctx.report_progress(80, 100)

    return a + b
```

### Create logs and progess with context

Throughout your tool function, call the `info()`, `warning()`, `debug()`, or `error()` methods to log **different types of messages** for the client. Also call the `report_progress()` method to estimate the amount of remaining work for the tool call.

```python
# server.py
@mcp.tool()
async def add(a: int, b: int, ctx: Context) -> int:
'''================================================='''
    await ctx.info("Preparing to add...")
    await ctx.report_progress(20, 100)

    await asyncio.sleep(2)

    await ctx.info("OK, adding...")
    await ctx.report_progress(80, 100)
'''================================================='''
    return a + b
```

### Define callbacks on the client

The client needs to define logging and progress callbacks, which will automatically be called whenever the server emits log or progress messages. These callbacks should try to display the provided logging and progress data to the user.

```python
# client.py
'''=================================================================='''
async def logging_callback(params: LoggingMessageNotificationParams):
    print(params.data)

async def print_progress_callback(
    progress: float, total: float | None, message: str | None
):
    if total is not None:
        percentage = (progress / total) * 100
        print(f"Progress: {progress}/{total} ({percentage:.1f}%)")
    else:
        print(f"Progress: {progress}")
'''=================================================================='''
```

### Pass callbacks to appropriate functions

Make sure you provide the **logging callback** to the `ClientSession` and the **progress callback** to the `call_tool()`function.

```python
# client.py
'''=================================================================='''
async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write, logging_callback=logging_callback
        ) as session:
            await session.initialize()

            await session.call_tool(
                name="add",
                arguments={"a": 1, "b": 3},
                progress_callback=print_progress_callback,
            )
'''=================================================================='''
```

## 5. Roots

Roots are a way to **grant MCP servers access to specific files and folders on your local machine**. Think of them as a permission system that says "Hey, MCP server, you can access these files" - but they do much more than just grant permission.

![](./figure/09/Roots_03.png)

### The Problem Roots Solve

Without roots, you'd run into a common issue. Imagine you have an MCP server with a video conversion tool that takes a file path and converts an MP4 to MOV format.

![](./figure/09/Roots_01.png)

When a user asks Claude to "convert biking.mp4 to mov format", Claude would call the tool with just the filename. But here's the problem : **Claude has no way to search through your entire file system to find where that file actually lives**.

![](./figure/09/Roots_02.png)

Your file system might be complex with files scattered across different directories. The user knows the `biking.mp4` file is in their Movies folder, but Claude doesn't have that context.

You could solve this by requiring users to always provide full paths, but that's not very user-friendly. Nobody wants to type out complete file paths every time.

### Roots in Action

Here's how the workflow changes with roots:

1. User asks to convert a video file
2. Claude calls `list_roots` to see what directories it can access
3. Claude calls `read_dir` on accessible directories to find the file
4. Once found, Claude calls the conversion tool with the full path

This happens automatically - users can still just say "convert biking.mp4" without providing full paths.

### Security and Boundaries

Roots also **provide security by limiting access**. If you only grant access to your `Desktop` folder, the MCP server cannot access files in other locations like `Documents` or `Downloads`.

When Claude tries to access a file outside the approved roots, it gets an error and can inform the user that the file isn't accessible from the current server configuration.
> 用 LLM 進行 limit 確定能完全限制嗎? 還是有方法可以 bypass?

### Implementation Details

The MCP SDK **doesn't automatically enforce root restrictions** - you need to implement this yourself. A typical pattern is to create a helper function like `is_path_allowed()` that:

- Takes a requested file path
- Gets the list of approved roots
- Checks if the requested path falls within one of those roots
- Returns true/false for access permission

You then call this function in any tool that accesses files or directories before performing the actual file operation.
> 要自己寫的話有沒有 recom example 可以參考，自己寫感覺很危險

### Key Benefits

- **User-friendly** : Users don't need to provide full file paths
- **Focused search** : Claude only looks in approved directories, making file discovery faster
- **Security** : Prevents accidental access to sensitive files outside approved areas
- **Flexibility** : You can provide roots through tools or inject them directly into prompts

Roots make MCP servers both more powerful and more secure by giving Claude the context it needs to find files while maintaining clear boundaries around what it can access.

## 6. Roots walkthrough
code 在 [./roots](./roots)

### Defining roots

Ideally, a user will dictate which files/folders can be accessed by the MCP server.

This program is set up to accept a list of CLI arguments, which are interpretted as paths that the user wants to allow access to.

That list of paths is provided to the `MCPClient` down on lines `42`.

```python
# main.py
async def main():
    claude_service = Claude(model=claude_model)
    '''================================================'''
    # Get root directories from command line arguments
    root_paths = sys.argv[1:]
    '''================================================'''
    if not root_paths:
        print("Usage: uv run main.py <root1> [root2] ...")
        print("Example: uv run main.py /path/to/videos /another/path")
        sys.exit(1)

    clients = {}

    async with AsyncExitStack() as stack:
        # Create the MCP client with the provided root directories
        doc_client = await stack.enter_async_context(
            MCPClient(
                command="uv", args=["run", "mcp_server.py"], roots=root_paths
            )
        )
        clients["doc_client"] = doc_client

        chat = CliChat(
            doc_client=doc_client,
            clients=clients,
            claude_service=claude_service,
        )

        cli = CliApp(chat)
        await cli.initialize()
        await cli.run()
```

### Creating root objects

According to the **MCP spec**, all roots should have a URI that begins with `file://`.

This function takes the list of paths of that the user provided and turns them into `Root` objects.

```python
# mcp_client.py
class MCPClient:
    def __init__(
        self,
        command: str,
        args: list[str],
        env: Optional[dict] = None,
        roots: Optional[list[str]] = None,
    ):
        self._command = command
        self._args = args
        self._env = env
        self._roots = self._create_roots(roots) if roots else []
        self._session: Optional[ClientSession] = None
        self._exit_stack: AsyncExitStack = AsyncExitStack()
    '''============================================================='''
    def _create_roots(self, root_paths: list[str]) -> list[Root]:
        """Convert path strings to Root objects."""
        roots = []
        for path in root_paths:
            p = Path(path).resolve()
            file_url = FileUrl(f"file://{p}")
            roots.append(Root(uri=file_url, name=p.name or "Root"))
        return roots
    '''============================================================='''
```

### Roots callback

The client doesn't immediately provide the list of roots to the server. Instead, the server can make a request to the client at some future point in time. We make a callback that will be executed when the server requests the roots. The callback needs to return the list of roots inside of a `ListRootsResult` object.

This callback is passed into the `ClientSession` down on line `58`.

```python
# mcp_client.py
'''====================================================='''
async def _handle_list_roots(
    self, context: RequestContext["ClientSession", None]
) -> ListRootsResult | ErrorData:
    """Callback for when server requests roots."""
    return ListRootsResult(roots=self._roots)
'''====================================================='''
```

### Using the roots

On to the server. The server will use the roots in two scenarios:

1. Whenever a tool attempts to access a file or folder
2. When a LLM (like Claude) needs to resolve a file or folder to a full path. Think of when a user says 'read the todos.txt file' - Claude needs to figure out where the text file is, and might do so by looking at the list of roots

To handle the second case, we can either **define a tool that lists out the roots** or **inject them directly in a prompt**.

```python
# mcp_server.py
'''============================================================================'''
@mcp.tool()
async def list_roots(ctx: Context):
    """
    List all directories that are accessible to this server.
    These are the root directories where files can be read from or written to.
    """
    roots_result = await ctx.session.list_roots()
    client_roots = roots_result.roots

    return [file_url_to_path(root.uri) for root in client_roots]
'''============================================================================'''
```

### Accessing the roots

Roots are accessed by calling `ctx.session.list_roots()`.

This sends a message back to the client, which causes it to run the root-listing callback.

```python
@mcp.tool()
async def list_roots(ctx: Context):
    """
    List all directories that are accessible to this server.
    These are the root directories where files can be read from or written to.
    """
    '''=============================================='''
    roots_result = await ctx.session.list_roots()
    client_roots = roots_result.roots
    '''=============================================='''

    return [file_url_to_path(root.uri) for root in client_roots]
```

### Authorizing access

> **Remember: the MCP SDK does not attempt to limit what files or folders your tools attempt to read! You must implement that check yourself.**

Consider implementing a function like `is_path_allowed`, which will decide whether a path is accessible by comparing it to the list of roots.

```python
# mcp_server.py
'''===================================================================='''
async def is_path_allowed(requested_path: Path, ctx: Context) -> bool:
    roots_result = await ctx.session.list_roots()
    client_roots = roots_result.roots

    if not requested_path.exists():
        return False

    if requested_path.is_file():
        requested_path = requested_path.parent

    for root in client_roots:
        root_path = file_url_to_path(root.uri)
        try:
            requested_path.relative_to(root_path)
            return True
        except ValueError:
            continue

    return False
'''===================================================================='''
```

### Authorizing access

Once you've put an authorization function together - like `is_path_allowed` - use it throughout your tools to ensure the requested path is accessible.

```python
# mcp_server.py
@mcp.tool()
async def convert_video(
    input_path: str = Field(description="Path to the input MP4 file"),
    format: str = Field(description="Output format (e.g. 'mov')"),
    *,
    ctx: Context,
):
    """Convert an MP4 video file to another format using ffmpeg"""
    input_file = VideoConverter.validate_input(input_path)

    '''=================================================================='''
    # Ensure the input file is contained in a root
    if not await is_path_allowed(input_file, ctx):
        raise ValueError(f"Access to path is not allowed: {input_path}")
    '''=================================================================='''

    return await VideoConverter.convert(input_path, format)
```

# Transports and communication

## 1. JSON message types

MCP (Model Context Protocol) uses **`JSON` messages** to handle communication between clients and servers. Understanding these message types is crucial for working with MCP, especially when dealing with different transport methods like the `streamable HTTP transport`.

### Message Format

**All MCP communication happens through `JSON` messages**. Each message type serves a specific purpose - whether it's calling a tool, listing available resources, or sending notifications about system events.

![](./figure/09/JSON_message_types_01.png)

Here's a typical example: when Claude needs to call a tool provided by an MCP server, the client sends a "Call Tool Request" message. The server processes this request, runs the tool, and responds with a "Call Tool Result" message containing the output.

![](./figure/09/JSON_message_types_02.png)

### MCP Specification

The complete list of message types is defined in the official MCP specification repository on GitHub. This specification is separate from the various SDK repositories (like Python or TypeScript SDKs) and serves as the authoritative source for how MCP should work.

The message types are written in `TypeScript` for convenience - not because they're executed as TypeScript code, but because TypeScript provides a **clear way** to describe data structures and types.

![](./figure/09/JSON_message_types_03.png)

### Message Categories

MCP messages fall into two main categories:

#### 1. **Request-Result Messages**
These messages always come in pairs. You send a request and expect to get a result back:

- Call Tool Request → Call Tool Result
- List Prompts Request → List Prompts Result
- Read Resource Request → Read Resource Result
- Initialize Request → Initialize Result

#### 2. **Notification Messages**
These are one-way messages that inform about events but don't require a response:

- Progress Notification - Updates on long-running operations
- Logging Message Notification - System log messages
- Tool List Changed Notification - When available tools change
- Resource Updated Notification - When resources are modified

![](./figure/09/JSON_message_types_04.png)

### Client vs Server Messages

The MCP specification organizes messages by who sends them:

Client messages include requests that clients send to servers (like tool calls) and notifications that clients might send.

Server messages include requests that servers send to clients and notifications that servers broadcast.

![](./figure/09/JSON_message_types_05.png)

### Why This Matters

Understanding that servers can send messages to clients is particularly important when working with different transport methods. Some transports, like the `streamable HTTP transport`, have **limitations on which types of messages can flow in which directions**.

The key insight is that MCP is designed as a bidirectional protocol - both clients and servers can initiate communication. This becomes crucial when you need to choose the right transport method for your specific use case.

## 2. The STDIO transport

MCP clients and servers communicate by exchanging `JSON` messages, but how do these messages actually get transmitted? The communication channel used is called a **transport**, and there are several ways to implement this - from `HTTP` requests to `WebSockets` to even writing `JSON` on a postcard (though that last one isn't recommended for production use).

### The Stdio Transport

When you're first developing an MCP server or client, the most commonly used transport is the `stdio transport`. This approach is straightforward: the client launches the MCP server as a subprocess and communicates through standard input and output streams.

Here's how it works:

1. Client sends messages to the server using the server's stdin
2. Server responds by writing to stdout
3. Either the server or client can send a message at any time
4. **Only works when client and server run on the same machine**

![](./figure/09/The_Stdio_Transport_01.png)

### Seeing Stdio in Action

You can actually test an MCP server directly from your terminal without writing a separate client. When you run a server with `uv run server.py`, it listens to `stdin` and writes responses to `stdout`. This means you can paste `JSON` messages directly into your terminal and see the server's responses immediately.

The terminal output shows the complete message exchange, including example messages for initialization and tool calls.

### MCP Connection Sequence

Every MCP connection must start with a specific three-message handshake:

1. **Initialize Request** : Client sends this first
2. **Initialize Result** : Server responds with capabilities
3. **Initialized Notification** : Client confirms (no response expected)

Only after this handshake can you send other requests like tool calls or prompt listings.

![](./figure/09/The_Stdio_Transport_02.png)

### Message Types and Flow

MCP supports various message types that flow in both directions:

![](./figure/09/The_Stdio_Transport_03.png)

The key insight is that **some messages require responses (requests → results) while others don't (notifications)**. **Both client and server can initiate communication at any time**.

### Four Communication Scenarios

With any transport, you need to handle four different communication patterns:

1. **Client → Server request** : Client writes to stdin
2. **Server → Client response** : Server writes to stdout
3. **Server → Client request** : Server writes to stdout
4. **Client → Server response** : Client writes to stdin

The beauty of stdio transport is its simplicity - either party can initiate communication at any time using these two channels.

![](./figure/09/The_Stdio_Transport_04.png)

### Why This Matters

Understanding stdio transport is crucial because it represents the "ideal" case where bidirectional communication is seamless. When we move to other transports like HTTP, we'll encounter limitations where the server cannot always initiate requests to the client. The stdio transport serves as our baseline for understanding what full MCP communication looks like before we tackle the constraints of other transport methods.

For development and testing, stdio transport is perfect. For production deployments where client and server need to run on different machines, you'll need to consider other transport options with their own trade-offs.

## 3. The StreamableHTTP transport

The streamable HTTP transport enables MCP clients to connect to remotely hosted servers over HTTP connections. Unlike the standard I/O transport that requires both client and server on the same machine, this transport opens up possibilities for public MCP servers that anyone can access.

![](./figure/09/The_StreamableHTTP_Transport_01.png)

However, there's an important caveat: some configuration settings can significantly limit your MCP server's functionality. If your application works perfectly with standard I/O transport locally but breaks when deployed with HTTP transport, this is likely the culprit.

![](./figure/09/The_StreamableHTTP_Transport_02.png)

### Configuration Settings That Matter

Two key settings control how the streamable HTTP transport behaves:

1. **stateless_http** : Controls connection state management
2. **json_response** : Controls response format handling

By default, both settings are **`false`**, but certain deployment scenarios may force you to set them to true. When enabled, these settings can break core functionality like *progress notifications*, *logging*, and *server-initiated requests*.

![](./figure/09/The_StreamableHTTP_Transport_03.png)
![](./figure/09/The_StreamableHTTP_Transport_04.png)
![](./figure/09/The_StreamableHTTP_Transport_05.png)
> ^ 這個跑不出來

### The HTTP Communication Challenge

To understand why these limitations exist, we need to review how HTTP communication works. In standard HTTP:

- Clients can easily initiate requests to servers (the server has a known URL)
- Servers can easily respond to these requests

![](./figure/09/The_StreamableHTTP_Transport_06.png)

- Servers **cannot** easily initiate requests to clients (clients don't have known URLs)
- Response patterns from client back to server become **problematic**

![](./figure/09/The_StreamableHTTP_Transport_07.png)

### MCP Message Types Affected

This HTTP limitation impacts specific MCP communication patterns. The following message types become difficult to implement with plain HTTP:

- **Server-initiated requests** : Create Message requests, List Roots requests
- **Notifications** : Progress notifications, Logging notifications, Initialized notifications, Cancelled notifications

These are exactly the features that break when you enable the restrictive HTTP settings. Progress bars disappear, logging stops working, and server-initiated sampling requests fail.

### The Streamable HTTP Solution

The streamable HTTP transport does provide a clever solution to work around HTTP's limitations, but it comes with trade-offs. When you're forced to use `stateless_http=True` or `json_response=True`, you're essentially telling the transport to operate within HTTP's constraints rather than working around them.

![](./figure/09/The_StreamableHTTP_Transport_08.png)

Understanding these limitations helps you make informed decisions about:

- Which transport to use for different deployment scenarios
- How to design your MCP server to gracefully handle HTTP constraints
- When to accept reduced functionality for the benefits of remote hosting

The key is knowing that these restrictions exist and planning your MCP server architecture accordingly. If your application heavily relies on **`server-initiated requests`** or **real-time notifications**, you may need to reconsider your transport choice or implement alternative communication patterns.

## 4. StreamableHTTP in depth

StreamableHTTP is MCP's solution to a fundamental problem: some MCP functionality requires the server to make requests to the client, but HTTP makes this challenging. Let's explore how StreamableHTTP works around this limitation and when you might need to break that workaround.

### The Core Problem

Some MCP features like sampling, notifications, and logging rely on the server initiating requests to the client. However, HTTP is designed for clients to make requests to servers, not the other way around. StreamableHTTP solves this with a clever workaround using **`Server-Sent Events (SSE)`**.

### How StreamableHTTP Works

The magic happens through a multi-step process that establishes persistent connections between client and server.

#### Initial Connection Setup
The process starts like any MCP connection:

1. Client sends an Initialize Request to the server
2. Server responds with an Initialize Result that includes a special mcp-session-id header
3. Client sends an Initialized Notification with the session ID

This session ID is crucial - it uniquely identifies the client and must be included in all future requests.

![](./figure/09/StreamableHTTP_in_Depth_01.png)

#### The SSE Workaround
After initialization, the client can make a `GET` request to establish a **`Server-Sent Events`** connection. This creates a **long-lived HTTP response** that the server can use to stream messages back to the client at any time.

This SSE connection is the key to allowing server-to-client communication. The server can now send requests, notifications, and other messages through this persistent channel.

![](./figure/09/StreamableHTTP_in_Depth_02.png)

### Tool Calls and Dual SSE Connections

When the client makes a tool call, things get more complex. The system creates two separate SSE connections:

- **Primary SSE Connection** : Used for server-initiated requests and stays open indefinitely
- **Tool-Specific SSE Connection** : Created for each tool call and closes automatically when the tool result is sent

![](./figure/09/StreamableHTTP_in_Depth_03.png)

#### Message Routing
Different types of messages get routed through different connections:

- **Progress notifications**: Sent through the **primary** `SSE` connection
- **Logging messages and tool results**: Sent through the **tool-specific** `SSE` connection

![](./figure/09/StreamableHTTP_in_Depth_04.png)

### Configuration Flags That Break the Workaround

StreamableHTTP includes two important configuration options:

1. **`stateless_http`**
2. **`json_response`**

Setting these to `True` can break the SSE workaround mechanism. You might want to enable these flags in certain scenarios, but doing so limits the full MCP functionality that depends on server-to-client communication.

### Key Takeaways

StreamableHTTP is more complex than other MCP transports because it has to work around HTTP's limitations. The SSE-based workaround enables full MCP functionality over HTTP, but understanding the dual-connection model is crucial for debugging and optimization.

When building MCP applications with StreamableHTTP, remember that session IDs are required for all requests after initialization, and the system automatically manages multiple SSE connections to handle different types of server-to-client communication.

## 5. State and the StreamableHTTP transport

The `stateless_http` and `json_response` flags in MCP servers control fundamental aspects of how your server behaves. Understanding when and why to use them is crucial, especially if you're planning to scale your server or deploy it in production.

### When You Need Stateless HTTP

Imagine you build an MCP server that becomes popular. Initially, you might have just a few clients connecting to a single server instance:

![](./figure/09/Stateless_HTTP_01.png)

As your server grows, you might have thousands of clients trying to connect. Running a single server instance won't scale to handle all that traffic:

![](./figure/09/Stateless_HTTP_02.png)

The typical solution is *horizontal* scaling - running multiple server instances behind a load balancer:

![](./figure/09/Stateless_HTTP_03.png)

But here's where things get complicated. Remember that **MCP clients** need two separate connections:

- A `GET` `SSE` connection for **receiving server-to-client requests**
- `POST` requests for **calling tools and receiving responses**

![](./figure/09/Stateless_HTTP_04.png)

With a load balancer, these requests might get *routed to different server instances*. If your tool needs to use Claude (through sampling), the server handling the `POST` request would need to coordinate with the server handling the `GET` `SSE` connection. This creates a complex coordination problem between servers.

![](./figure/09/Stateless_HTTP_05.png)

### How Stateless HTTP Solves This

Setting `stateless_http=True` eliminates this coordination problem, but with significant trade-offs:

When stateless HTTP is enabled:

1. Clients don't get session IDs -> the server can't track individual clients
2. No server-to-client requests -> the `GET` `SSE` pathway becomes unavailable
3. No sampling -> can't use Claude or other AI models
4. No progress reports -> can't send progress updates during long operations
5. No subscriptions -> can't notify clients about resource updates

However, there's one benefit: client initialization is no longer required. Clients can make requests directly without the initial handshake process.

![](./figure/09/Stateless_HTTP_06.png)

### Understanding JSON Response

The `json_response=True` flag is simpler - it just **disables** streaming for `POST` request responses. Instead of getting multiple `SSE` messages as a tool executes, you get **only the final result as plain JSON**.

With streaming disabled:

1. No intermediate progress messages
2. No log statements during execution
3. Just the final tool result

### When to Use These Flags

Use stateless HTTP when:

1. You need horizontal scaling with load balancers
2. You don't need server-to-client communication
3. Your tools don't require AI model sampling
4. You want to minimize connection overhead

Use JSON response when:

1. You don't need streaming responses
2. You prefer simpler, non-streaming HTTP responses
3. You're integrating with systems that expect plain JSON

### Development vs Production

If you're developing locally with standard I/O transport but planning to deploy with HTTP transport, test with the same transport you'll use in production. The behavior differences between stateful and stateless modes can be significant, and it's better to catch any issues during development rather than after deployment.

These flags fundamentally change how your MCP server operates, so choose them based on your specific scaling and functionality requirements.

# Extra

根據來源文件 [MCP transport doc](https://modelcontextprotocol.io/specification/2025-11-25/basic/transports#streamable-http)，對於 **Streamable HTTP** 傳輸機制，確實詳細規範了客戶端（Client）與伺服器（Server）在不同情況下必須使用 **POST** 或 **GET** 的時機：

### 1. 客戶端 (Client) 的規範

*   **必須使用 POST 的情況：**
    *   **發送所有訊息**：客戶端發送給伺服器的**每一條** JSON-RPC 訊息（包括請求、通知或回應），都**必須**是一個新的 HTTP **POST** 請求。
    *   **初始化請求**：在建立連線的初始階段，客戶端必須發送一個 POST 請求來包含 `InitializeRequest`。
*   **使用 GET 的情況：**
    *   **開啟監聽串流**：客戶端**可以**發送一個 HTTP **GET** 請求來開啟一個 SSE (Server-Sent Events) 串流，以便在不先發送資料的情況下接收伺服器主動發送的訊息。
    *   **恢復連線**：若連線因網路問題或伺服器關閉而中斷，客戶端**應**發送一個 **GET** 請求並包含 `Last-Event-ID` 標頭來恢復串流。無論原始串流是透過 POST 還是 GET 啟動的，**恢復操作一律使用 HTTP GET**。

### 2. 伺服器 (Server) 的規範

伺服器並不主動發起 POST 或 GET 請求，而是根據客戶端的請求進行回應：

*   **針對 POST 請求的回應：**
    *   若接收到客戶端的回應或通知，且伺服器接受該輸入，則必須回傳 **202 Accepted**（無主體）。
    *   若接收到客戶端的請求，伺服器可以選擇回傳單一的 JSON 物件（`application/json`），或啟動一個 SSE 串流（`text/event-stream`）來傳回多條訊息。
*   **針對 GET 請求的回應：**
    *   伺服器必須回傳 `text/event-stream` 以開啟串流，若不支援則回傳 **405 Method Not Allowed**。
*   **在串流中發送訊息的限制：**
    *   **GET 串流限制**：在由 GET 啟動的串流中，伺服器**不得**發送 JSON-RPC **回應 (response)**，除非是在恢復（Resuming）一個先前與客戶端請求相關聯的串流時。
    *   伺服器可以在 POST 啟動的 SSE 串流中發送與該請求相關的請求或通知，並在最後發送回應。

### 3. 其他方法
*   **DELETE**：當客戶端不再需要特定會話（Session）時，**應**向 MCP 端點發送一個 HTTP **DELETE** 請求來明確終止該會話。

總結來說，**POST 是訊息傳輸的主要通道（Client 向 Server）**，而 **GET 主要用於建立或恢復接收訊息的 SSE 通道（Server 向 Client）**。